In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder.appName("shuffle-practice").master("local[*]").getOrCreate()
print(spark.version)

4.2.0


In [3]:
bookings = spark.createDataFrame(
    [(1, "ORD-LHR", "Economy", 640.0),
     (2, "ORD-LHR", "Business", 2900.0),
     (3, "ORD-FRA", "Economy", 590.0),
     (4, "ORD-FRA", "Economy", 610.0),
     (5, "ORD-CDG", "Business", 3100.0)],
    ["booking_id", "route", "cabin", "fare"])

bookings.show()

+----------+-------+--------+------+
|booking_id|  route|   cabin|  fare|
+----------+-------+--------+------+
|         1|ORD-LHR| Economy| 640.0|
|         2|ORD-LHR|Business|2900.0|
|         3|ORD-FRA| Economy| 590.0|
|         4|ORD-FRA| Economy| 610.0|
|         5|ORD-CDG|Business|3100.0|
+----------+-------+--------+------+



In [4]:
aggdf = bookings.groupBy("route").agg(sum("fare").alias("revenue"), count("*").alias("bookings"))

In [5]:
aggdf.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[route#1], functions=[sum(fare#3), count(1)])
   +- Exchange hashpartitioning(route#1, 200), ENSURE_REQUIREMENTS, [plan_id=27]
      +- HashAggregate(keys=[route#1], functions=[partial_sum(fare#3), partial_count(1)])
         +- Project [route#1, fare#3]
            +- Scan ExistingRDD[booking_id#0L,route#1,cabin#2,fare#3]




In [7]:
aggdf.show()

+-------+-------+--------+
|  route|revenue|bookings|
+-------+-------+--------+
|ORD-LHR| 3540.0|       2|
|ORD-FRA| 1200.0|       2|
|ORD-CDG| 3100.0|       1|
+-------+-------+--------+



In [10]:
big = (spark.range(0, 5_000_000)
       .withColumn("bucket", col("id") % 100)
       .groupBy("bucket")
       .count())
big.collect()

[Row(bucket=26, count=50000),
 Row(bucket=29, count=50000),
 Row(bucket=65, count=50000),
 Row(bucket=19, count=50000),
 Row(bucket=54, count=50000),
 Row(bucket=0, count=50000),
 Row(bucket=22, count=50000),
 Row(bucket=7, count=50000),
 Row(bucket=77, count=50000),
 Row(bucket=34, count=50000),
 Row(bucket=50, count=50000),
 Row(bucket=94, count=50000),
 Row(bucket=57, count=50000),
 Row(bucket=32, count=50000),
 Row(bucket=43, count=50000),
 Row(bucket=84, count=50000),
 Row(bucket=31, count=50000),
 Row(bucket=39, count=50000),
 Row(bucket=98, count=50000),
 Row(bucket=25, count=50000),
 Row(bucket=95, count=50000),
 Row(bucket=71, count=50000),
 Row(bucket=6, count=50000),
 Row(bucket=68, count=50000),
 Row(bucket=72, count=50000),
 Row(bucket=87, count=50000),
 Row(bucket=58, count=50000),
 Row(bucket=9, count=50000),
 Row(bucket=27, count=50000),
 Row(bucket=63, count=50000),
 Row(bucket=51, count=50000),
 Row(bucket=56, count=50000),
 Row(bucket=52, count=50000),
 Row(bucket=17

In [11]:
spark.sparkContext.defaultParallelism


8

In [12]:
import os
print(os.listdir("/home/jovyan/work"))

['shuffle-practice.ipynb', '.ipynb_checkpoints']
